# Notebook 10 — Model D Quick Test (Threshold = 50 Reviews)
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Quick test to see if training Model D on the expanded 50-review threshold
dataset improves Macro F1 over the current production baseline.

**If this shows improvement:** Rerun NB03 through NB07 with MIN_REVIEWS=50.
**If no improvement:** Keep original dataset. No further action needed.

**Nothing is saved to disk. This is a read-only test.**

**Inputs (read-only):**
- `data/processed/games_features_t4.csv` — original 100-review dataset
- `data/processed/games_features_t4_threshold50.csv` — new 50-review dataset
- `outputs/results/03_winner_hyperparameters.json` — current tuned CatBoost hyperparameters
- `outputs/results/tier_ablation.csv` — current production baseline (Model D, T4)

---
**Revision note (v2):** Two bugs found and fixed, both the same classes already found and
fixed in notebooks 04, 05, 07, 08 and 09:

1. **Hardcoded stale hyperparameters/baseline.** `CB_PARAMS` hardcoded the pre-fix
   values (`depth=8, lr=0.05, l2=3`) instead of the current tuned ones
   (`depth=4, lr=0.1, l2=5`), and `BASELINE_MACRO_F1`/`BASELINE_MINOR_F1`/`BASELINE_ACCURACY`
   were hardcoded to stale numbers (`0.6690`/`0.5172`/`0.7385`) — the `0.7385` was actually
   a mislabeled AUC value, not real accuracy. Now loaded live from
   `03_winner_hyperparameters.json` and `tier_ablation.csv`.
2. **Feature-count mismatch (54 vs 53 features).** Section 2's column-compatibility check
   excluded only `TARGET_COL` and `short_description`, so `dlc_count` was accidentally left
   in as a 54th feature. Every other notebook in the pipeline (NB03/04/05) excludes
   `dlc_count` from modelling — it's kept in the CSV only because it feeds
   `price_to_dlc_ratio`, not because it's used standalone (r=0.005, not significant). This
   meant NB10's "Orig (>=100 reviews)" run wasn't actually reproducing production Model D —
   it was a different, 54-feature model — so the delta reported against the true baseline
   wasn't apples-to-apples. Fixed to exclude `dlc_count` too, matching production exactly
   (53 features, same as NB03/04/05).

Both fixes are purely in Sections 1–2; the training/comparison/verdict logic in Sections
4–6 is unchanged (it already reads from the fixed variables, not literals).

---
## Contents
1. Setup and Imports
2. Load Both Datasets
3. Define Feature Sets and Splits
4. Train and Evaluate Model D
5. Head-to-Head Comparison
6. Verdict

## 1. Setup and Imports

In [1]:
import sys
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score,
                              classification_report)

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TARGET_COL, COL_DESC, COL_DLC, RANDOM_STATE,
    TOP_10_GENRES, TOP_20_TAGS, KEY_CATEGORIES
)

RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results'

# ── v2 fix: load current tuned CatBoost hyperparameters live ─────────────────
# (was hardcoded to stale depth=8/lr=0.05/l2=3 — same bug already fixed in NB04/05/07/08/09)
hp_path = RESULTS_DIR / '03_winner_hyperparameters.json'
with open(hp_path) as f:
    _hp = json.load(f)['hyperparameters']

CB_PARAMS = {
    'depth':             _hp['depth'],
    'learning_rate':     _hp['learning_rate'],
    'iterations':        _hp['iterations'],
    'l2_leaf_reg':       _hp['l2_leaf_reg'],
    'scale_pos_weight':  _hp['scale_pos_weight'],
    'random_seed':       RANDOM_STATE,
    'verbose':           0,
    'allow_writing_files': False
}
print(f'CatBoost params loaded from {hp_path.name}:')
print(f'  depth={CB_PARAMS["depth"]}, lr={CB_PARAMS["learning_rate"]}, '
      f'iterations={CB_PARAMS["iterations"]}, l2={CB_PARAMS["l2_leaf_reg"]}, '
      f'scale_pos_weight={CB_PARAMS["scale_pos_weight"]}')

# ── v2 fix: load current production baseline (Model D, T4) live from tier_ablation.csv ──
# (was hardcoded to stale/mislabeled 0.6690 / 0.5172 / 0.7385 — 0.7385 was actually
# a mislabeled AUC value, not real accuracy)
tier_path = RESULTS_DIR / 'tier_ablation.csv'
_tier = pd.read_csv(tier_path)
_d_row = _tier[(_tier['model'] == 'Model D') & (_tier['tier'] == 'T4')].iloc[0]

BASELINE_MACRO_F1  = float(_d_row['test_macro_f1'])
BASELINE_MINOR_F1  = float(_d_row['test_minority_f1'])
BASELINE_ACCURACY  = float(_d_row['test_accuracy'])
BASELINE_AUC       = float(_d_row['test_auc'])

print(f'\nImports OK')
print(f'Project root : {PROJECT_ROOT}')
print(f'Random state : {RANDOM_STATE}')
print(f'Baseline     : Model D Macro F1 = {BASELINE_MACRO_F1} '
      f'(from {tier_path.name}, T4 — produced by NB04)')

CatBoost params loaded from 03_winner_hyperparameters.json:
  depth=4, lr=0.1, iterations=1000, l2=5, scale_pos_weight=0.5

Imports OK
Project root : C:\Users\3214h\Documents\fyp-steam-reception
Random state : 42
Baseline     : Model D Macro F1 = 0.6603 (from tier_ablation.csv, T4 — produced by NB04)


## 2. Load Both Datasets

In [2]:
# ── Original dataset (100-review threshold) ──────────────────────────────────
orig_path = PROJECT_ROOT / 'data' / 'processed' / 'games_features_t4.csv'
assert orig_path.exists(), f'Original dataset not found: {orig_path}'
df_orig = pd.read_csv(orig_path)
print(f'Original (>=100 reviews) : {len(df_orig):,} rows x {df_orig.shape[1]} columns')
print(f'  Class balance: {df_orig[TARGET_COL].value_counts(normalize=True).round(3).to_dict()}')

# ── New dataset (50-review threshold) ─────────────────────────────────────────
new_path = PROJECT_ROOT / 'data' / 'processed' / 'games_features_t4_threshold50.csv'
assert new_path.exists(), f'New dataset not found: {new_path}'
df_new = pd.read_csv(new_path)
print(f'\nNew (>=50 reviews)       : {len(df_new):,} rows x {df_new.shape[1]} columns')
print(f'  Class balance: {df_new[TARGET_COL].value_counts(normalize=True).round(3).to_dict()}')

# ── Confirm column structure is compatible ─────────────────────────────────────
# v2 fix: also exclude dlc_count, matching production Model D exactly (NB03/04/05 all
# exclude it — it's kept in the CSV only to feed price_to_dlc_ratio, not used standalone).
# The original notebook excluded only TARGET_COL and short_description, which left
# dlc_count in as an accidental 54th feature (the same bug flagged for NB11).
EXCLUDE_COLS = [TARGET_COL, COL_DESC, COL_DLC]
orig_feat_cols = [c for c in df_orig.columns if c not in EXCLUDE_COLS]
new_feat_cols  = [c for c in df_new.columns  if c not in EXCLUDE_COLS]

common = set(orig_feat_cols) & set(new_feat_cols)
only_orig = set(orig_feat_cols) - set(new_feat_cols)
only_new  = set(new_feat_cols)  - set(orig_feat_cols)

print(f'\nColumn compatibility (dlc_count excluded — matches production Model D):')
print(f'  Common feature columns   : {len(common)}')
print(f'  Only in original         : {sorted(only_orig) if only_orig else "none"}')
print(f'  Only in new dataset      : {sorted(only_new)  if only_new  else "none"}')

# Use only the intersection so Model D trains on the same features in both cases
FEATURE_COLS = sorted(common)
print(f'\nFeatures used for both Model D tests: {len(FEATURE_COLS)} '
      f'(expected 53, matching production Model D)')
if len(FEATURE_COLS) != 53:
    print(f'  WARNING: expected 53 features (production Model D) but got {len(FEATURE_COLS)} — '
          f'investigate before trusting the comparison below.')

Original (>=100 reviews) : 20,383 rows x 55 columns
  Class balance: {1: 0.718, 0: 0.282}

New (>=50 reviews)       : 27,750 rows x 56 columns
  Class balance: {1: 0.685, 0: 0.315}

Column compatibility (dlc_count excluded — matches production Model D):
  Common feature columns   : 53
  Only in original         : none
  Only in new dataset      : none

Features used for both Model D tests: 53 (expected 53, matching production Model D)


## 3. Define Feature Sets and Splits

Both datasets use the same RANDOM_STATE and test_size so results are comparable.
Note: the test sets will differ in size since the datasets have different row counts,
but both are stratified so the class balance is preserved.

In [3]:
# ── Original dataset split ────────────────────────────────────────────────────
X_orig = df_orig[FEATURE_COLS].values
y_orig = df_orig[TARGET_COL].values

X_tr_orig, X_te_orig, y_tr_orig, y_te_orig = train_test_split(
    X_orig, y_orig,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_orig
)

# ── New dataset split ──────────────────────────────────────────────────────────
X_new = df_new[FEATURE_COLS].values
y_new = df_new[TARGET_COL].values

X_tr_new, X_te_new, y_tr_new, y_te_new = train_test_split(
    X_new, y_new,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_new
)

print('Dataset splits:')
print(f'  Original — train: {len(y_tr_orig):,}  test: {len(y_te_orig):,}  '
      f'(positive rate: {y_tr_orig.mean()*100:.1f}%)')
print(f'  New      — train: {len(y_tr_new):,}  test: {len(y_te_new):,}  '
      f'(positive rate: {y_tr_new.mean()*100:.1f}%)')
print(f'\nAdditional training games: {len(y_tr_new) - len(y_tr_orig):,}')
print(f'Additional training Not Well Received: '
      f'{(y_tr_new==0).sum() - (y_tr_orig==0).sum():,}')

Dataset splits:
  Original — train: 16,306  test: 4,077  (positive rate: 71.8%)
  New      — train: 22,200  test: 5,550  (positive rate: 68.5%)

Additional training games: 5,894
Additional training Not Well Received: 2,397


## 4. Train and Evaluate Model D

5-fold CV on training set for a reliable performance estimate,
then final evaluation on the held-out test set.

In [4]:
def evaluate_model_d(X_train, y_train, X_test, y_test, label):
    """
    Train Model D and evaluate on the test set.
    Returns a dict of all metrics.
    """
    print(f'--- {label} ---')

    # 5-fold CV on training set
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_f1s = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        m = CatBoostClassifier(**CB_PARAMS)
        m.fit(X_train[tr_idx], y_train[tr_idx])
        pred = m.predict(X_train[val_idx])
        cv_f1s.append(f1_score(y_train[val_idx], pred, average='macro'))
        print(f'  CV fold {fold}/5 — Macro F1: {cv_f1s[-1]:.4f}', end='\r')
    print()

    cv_mean = np.mean(cv_f1s)
    cv_std  = np.std(cv_f1s)
    print(f'  CV Macro F1 : {cv_mean:.4f} ± {cv_std:.4f}')

    # Final model on full training set
    model = CatBoostClassifier(**CB_PARAMS)
    model.fit(X_train, y_train)

    # Test set evaluation
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    macro = f1_score(y_test, y_pred, average='macro')
    minor = f1_score(y_test, y_pred, pos_label=0)
    major = f1_score(y_test, y_pred, pos_label=1)
    acc   = accuracy_score(y_test, y_pred)
    auc   = roc_auc_score(y_test, y_prob)

    print(f'  Test Macro F1    : {macro:.4f}')
    print(f'  Test Minority F1 : {minor:.4f}')
    print(f'  Test Majority F1 : {major:.4f}')
    print(f'  Test Accuracy    : {acc:.4f}')
    print(f'  Test AUC-ROC     : {auc:.4f}')
    print()

    return {
        'label':     label,
        'cv_mean':   cv_mean,
        'cv_std':    cv_std,
        'macro_f1':  macro,
        'minor_f1':  minor,
        'major_f1':  major,
        'accuracy':  acc,
        'auc':       auc,
    }

# ── Run both evaluations ──────────────────────────────────────────────────────
print('Training Model D on ORIGINAL dataset (>=100 reviews, 16,306 training games)...')
res_orig = evaluate_model_d(X_tr_orig, y_tr_orig, X_te_orig, y_te_orig,
                             'Model D — Original (>=100 reviews)')

print('Training Model D on NEW dataset (>=50 reviews, 22,200 training games)...')
res_new  = evaluate_model_d(X_tr_new, y_tr_new, X_te_new, y_te_new,
                             'Model D — New (>=50 reviews)')

Training Model D on ORIGINAL dataset (>=100 reviews, 16,306 training games)...
--- Model D — Original (>=100 reviews) ---
  CV fold 5/5 — Macro F1: 0.6658
  CV Macro F1 : 0.6647 ± 0.0041
  Test Macro F1    : 0.6572
  Test Minority F1 : 0.5159
  Test Majority F1 : 0.7985
  Test Accuracy    : 0.7155
  Test AUC-ROC     : 0.7402

Training Model D on NEW dataset (>=50 reviews, 22,200 training games)...
--- Model D — New (>=50 reviews) ---
  CV fold 5/5 — Macro F1: 0.6583
  CV Macro F1 : 0.6615 ± 0.0032
  Test Macro F1    : 0.6590
  Test Minority F1 : 0.5591
  Test Majority F1 : 0.7589
  Test Accuracy    : 0.6883
  Test AUC-ROC     : 0.7400



## 5. Head-to-Head Comparison

In [5]:
print('=' * 72)
print('HEAD-TO-HEAD COMPARISON')
print('=' * 72)
print()
print(f'  {"Metric":<25} {"NB04 Baseline":>15} {"Orig (>=100)":>15} {"New (>=50)":>12} {"Change":>8}')
print('  ' + '-' * 78)

metrics = [
    ('CV Macro F1',    'cv_mean',  None),
    ('CV Std',         'cv_std',   None),
    ('Test Macro F1',  'macro_f1', BASELINE_MACRO_F1),
    ('Test Minority F1','minor_f1', BASELINE_MINOR_F1),
    ('Test Majority F1','major_f1', None),
    ('Test Accuracy',  'accuracy', BASELINE_ACCURACY),
    ('Test AUC-ROC',   'auc',      BASELINE_AUC),
]

for label, key, baseline in metrics:
    orig_val = res_orig[key]
    new_val  = res_new[key]
    diff     = new_val - orig_val
    base_str = f'{baseline:.4f}' if baseline is not None else '—'
    print(f'  {label:<25} {base_str:>15} {orig_val:>15.4f} {new_val:>12.4f} {diff:>+8.4f}')

print()

# Summary of gains
macro_gain = res_new['macro_f1'] - res_orig['macro_f1']
minor_gain = res_new['minor_f1'] - res_orig['minor_f1']
vs_nb04    = res_new['macro_f1'] - BASELINE_MACRO_F1

print(f'  New vs Original  : Macro F1 {macro_gain:+.4f}  |  Minority F1 {minor_gain:+.4f}')
print(f'  New vs NB04      : Macro F1 {vs_nb04:+.4f}')

HEAD-TO-HEAD COMPARISON

  Metric                      NB04 Baseline    Orig (>=100)   New (>=50)   Change
  ------------------------------------------------------------------------------
  CV Macro F1                             —          0.6647       0.6615  -0.0032
  CV Std                                  —          0.0041       0.0032  -0.0009
  Test Macro F1                      0.6603          0.6572       0.6590  +0.0018
  Test Minority F1                   0.5214          0.5159       0.5591  +0.0433
  Test Majority F1                        —          0.7985       0.7589  -0.0396
  Test Accuracy                      0.7172          0.7155       0.6883  -0.0272
  Test AUC-ROC                       0.7383          0.7402       0.7400  -0.0002

  New vs Original  : Macro F1 +0.0018  |  Minority F1 +0.0433
  New vs NB04      : Macro F1 -0.0013


## 6. Verdict

In [6]:
print('=' * 72)
print('VERDICT')
print('=' * 72)
print()

macro_gain = res_new['macro_f1'] - BASELINE_MACRO_F1
minor_gain = res_new['minor_f1'] - BASELINE_MINOR_F1

if res_new['macro_f1'] > BASELINE_MACRO_F1 + 0.005:
    print(f'CLEAR IMPROVEMENT')
    print(f'The 50-review threshold dataset gives Macro F1 = {res_new["macro_f1"]:.4f}')
    print(f'vs the current best of {BASELINE_MACRO_F1} ({macro_gain:+.4f} gain).')
    print()
    print('Recommendation: PROCEED with full rerun.')
    print('  1. Update src/config.py  MIN_REVIEW_COUNT = 50')
    print('  2. Rerun NB02 to regenerate games_features_t4.csv with new threshold')
    print('  3. Rerun NB03 through NB07 — all results will update automatically')
    print('  4. Rerun NB05 on Colab to retrain Model E on the expanded dataset')
    print('  5. Update thesis results tables with new numbers')

elif res_new['macro_f1'] > BASELINE_MACRO_F1:
    print(f'MARGINAL IMPROVEMENT')
    print(f'The 50-review threshold dataset gives Macro F1 = {res_new["macro_f1"]:.4f}')
    print(f'vs the current best of {BASELINE_MACRO_F1} ({macro_gain:+.4f} gain).')
    print()
    print('Recommendation: Borderline — consider your submission timeline.')
    print('  The gain is real but small. A full rerun would take significant time.')
    print('  If time permits, proceed. If close to submission, document this')
    print('  as a future work item — the finding itself is a contribution.')

elif res_new['macro_f1'] == BASELINE_MACRO_F1 or abs(macro_gain) <= 0.002:
    print(f'NO MEANINGFUL IMPROVEMENT')
    print(f'The 50-review threshold dataset gives Macro F1 = {res_new["macro_f1"]:.4f}')
    print(f'vs the current best of {BASELINE_MACRO_F1} (difference: {macro_gain:+.4f}).')
    print()
    print('Recommendation: Keep original dataset. The 50-99 review games add volume')
    print('but not enough signal to move the performance needle meaningfully.')
    print('Document the experiment and its finding in the thesis.')

else:
    print(f'PERFORMANCE DEGRADED')
    print(f'The 50-review threshold dataset gives Macro F1 = {res_new["macro_f1"]:.4f}')
    print(f'vs the current best of {BASELINE_MACRO_F1} ({macro_gain:+.4f}).')
    print()
    print('Recommendation: Do NOT proceed. Games with 50-99 reviews have noisy labels.')
    print('The additional games are hurting more than helping.')
    print('Keep the 100-review threshold. Document the experiment as a negative finding.')

print()
print(f'Minority F1 change: {minor_gain:+.4f}  (catching Not Well Received games)')
if minor_gain > 0.01:
    print('  The model is noticeably better at identifying games that will receive poor reception.')
elif minor_gain > 0:
    print('  Small improvement in catching Not Well Received games.')
elif minor_gain < -0.01:
    print('  The model is worse at catching Not Well Received games despite more training examples.')
    print('  The extra 50-99 review games may have noisy labels that confused the model.')

VERDICT

NO MEANINGFUL IMPROVEMENT
The 50-review threshold dataset gives Macro F1 = 0.6590
vs the current best of 0.6603 (difference: -0.0013).

Recommendation: Keep original dataset. The 50-99 review games add volume
but not enough signal to move the performance needle meaningfully.
Document the experiment and its finding in the thesis.

Minority F1 change: +0.0377  (catching Not Well Received games)
  The model is noticeably better at identifying games that will receive poor reception.
